In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
ket1 = np.array([0.0 + 0.0j, 1.0 + 0.0j])


def normalize(state: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(state)
    if norm == 0:
        raise ValueError("Statevector cannot be the zero vector.")
    return state / norm


alpha = 1 / np.sqrt(2)
beta = 1j / np.sqrt(2)
psi = normalize(alpha * ket0 + beta * ket1)
probabilities = np.abs(psi) ** 2

print("statevector:", psi)
print("probabilities:", probabilities)
print("sum of probabilities:", probabilities.sum())

In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
I = np.eye(2, dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
CNOT = np.array(
    [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ],
    dtype=complex,
)

ket00 = np.kron(ket0, ket0)
bell = CNOT @ np.kron(H, I) @ ket00

print("Bell state:", bell)
print("Probabilities:", np.abs(bell) ** 2)
print("Norm:", np.linalg.norm(bell))

In [ ]:
import numpy as np

I = np.eye(2, dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)


def ry(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2), np.cos(theta / 2)],
        ],
        dtype=complex,
    )


ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
psi = ket0.copy()

psi = H @ psi
psi = ry(np.pi / 3) @ psi
psi = Z @ psi

print("final state:", psi)
print("norm:", np.linalg.norm(psi))
print("probabilities:", np.abs(psi) ** 2)

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
Z = np.array([[1, 0], [0, -1]], dtype=complex)


def sample_pauli_z(
    state: np.ndarray, shots: int, rng: np.random.Generator
) -> np.ndarray:
    probabilities = np.abs(state) ** 2
    return rng.choice(np.array([1, -1]), size=shots, p=probabilities)


psi = np.array([np.sqrt(0.8) + 0.0j, np.sqrt(0.2) + 0.0j])
exact_expectation = np.vdot(psi, Z @ psi).real
samples = sample_pauli_z(psi, shots=1000, rng=rng)
estimated_expectation = samples.mean()

print("Exact <Z>:", exact_expectation)
print("Estimated <Z> from 1000 shots:", estimated_expectation)

In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
Z = np.array([[1, 0], [0, -1]], dtype=complex)


def ry(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2), np.cos(theta / 2)],
        ],
        dtype=complex,
    )


def quantum_score(x: float, theta: float) -> float:
    state = ry(theta) @ (ry(x) @ ket0)
    return float(np.vdot(state, Z @ state).real)


def predict_probability(x: float, theta: float) -> float:
    return 0.5 * (quantum_score(x, theta) + 1.0)


def loss(xs: np.ndarray, ys: np.ndarray, theta: float) -> float:
    preds = np.array([predict_probability(x, theta) for x in xs])
    return float(np.mean((preds - ys) ** 2))


def gradient_parameter_shift(
    xs: np.ndarray, ys: np.ndarray, theta: float
) -> float:
    preds = np.array([predict_probability(x, theta) for x in xs])
    shifted_plus = np.array([
        quantum_score(x, theta + np.pi / 2) for x in xs
    ])
    shifted_minus = np.array([
        quantum_score(x, theta - np.pi / 2) for x in xs
    ])

    dscore_dtheta = 0.5 * (shifted_plus - shifted_minus)
    dpred_dtheta = 0.5 * dscore_dtheta
    grad = np.mean(2.0 * (preds - ys) * dpred_dtheta)
    return float(grad)


xs = np.linspace(0.0, np.pi, 50)
ys = (xs < (np.pi / 2)).astype(float)

theta = 0.3
learning_rate = 0.2
for epoch in range(60):
    grad = gradient_parameter_shift(xs, ys, theta)
    theta -= learning_rate * grad

    if epoch % 10 == 0 or epoch == 59:
        print(
            f"epoch={epoch:02d} theta={theta:.4f} "
            f"loss={loss(xs, ys, theta):.6f}"
        )